# 00-2. Preprocessing (CPTAC-PDAC)

In [2]:
library(data.table)
library(vespa)

In [2]:
if (!dir.exists("./data/cptac-pdac")) {
  out <- system2(
    "bash",
    "./tools/scripts/cptac-pdac.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

In [3]:
if (!file.exists("./tools/references/library.fasta")) {
  out <- system2(
    "bash",
    "./tools/scripts/fasta.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

## Import data

In [45]:
phospho <- fread("./data/cptac-pdac/phosphoproteomics_site_level_MD_abundance_tumor.cct")

In [46]:
phospho

Index,Gene,Peptide,C3N-03884,C3L-03123,C3L-01687,C3L-00589,C3L-00599,C3L-01054,C3L-03356,⋯,C3N-01375,C3N-01381,C3L-00401,C3L-02118,C3N-00511,C3L-02613,C3N-00512,C3L-02899,C3N-03006,C3N-03069
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NP_000005.2_S928,A2M,ETTFNSLLCPSGGEVsEELSLK,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NP_000009.1_S489,ACADVL,ELSGLGsALK,13.84222,13.64339,14.25921,13.97222,13.83410,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NP_000010.1_Y170,ACAT1,GSTPyGGVK,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NP_000011.2_S160,ACVRL1,GLHSELGEsSLILK,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,18.01134,18.82855,18.32407,18.50001,18.10603,18.59428
NP_000011.2_S161,ACVRL1,GLHSELGESsLILK,NA,NA,NA,NA,NA,NA,NA,⋯,18.96619,18.31143,17.44497,18.68298,NA,NA,NA,NA,NA,NA
NP_000011.2_S495,ACVRL1,ISNsPEKPK,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NP_000012.1_S366,PSEN1,AAVQELSsSILAGEDPEER,NA,NA,NA,NA,NA,16.51506,16.11122,⋯,16.73622,16.74612,17.00373,16.51724,NA,NA,NA,NA,NA,NA
NP_000012.1_S367,PSEN1,AAVQELSSsILAGEDPEER,18.93493,18.84214,18.73961,18.76958,17.95985,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NP_000012.1_S43,PSEN1,sLGHPEPLSNGRPQGNSR,NA,NA,NA,NA,NA,19.56398,20.97608,⋯,21.15073,20.68819,19.97195,21.41541,21.54934,22.31834,20.99347,21.68634,21.15257,21.09441


In [47]:
sample_cols <- setdiff(
  names(phospho),
  c("Index", "Gene", "Peptide")
)

for (j in sample_cols) {
  set(
    phospho,
    i = which(!is.na(phospho[[j]]) & !is.finite(phospho[[j]])),
    j = j,
    value = NA_real_
  )
}

phospho[, row_id := .I]
phospho[, gene_id := Gene]
phospho[, refseq_id := sub("_[STY][0-9]+$", "", Index)]
phospho[, refseq_phosphosite := sub("^.*_([STY][0-9]+)$", "\\1", Index)]

phospho[, modified_peptide_sequence := Peptide]
phospho[, peptide_sequence := toupper(Peptide)]

In [48]:
phospho_pep <- phospho[
  ,
  .(
    peptide_candidate = unlist(strsplit(Peptide, ";", fixed = TRUE))
  ),
  by = .(
    row_id,
    gene_id,
    refseq_id,
    refseq_phosphosite
  )
]

phospho_pep[, peptide_candidate := trimws(peptide_candidate)]
phospho_pep <- phospho_pep[peptide_candidate != ""]

phospho_pep[
  ,
  n_mods := lengths(
    regmatches(
      peptide_candidate,
      gregexpr("[sty]", peptide_candidate)
    )
  )
]

table(phospho_pep$n_mods)


    1     2     3 
50714  6409  3168 

In [49]:
phospho_pep <- phospho_pep[n_mods == 1]

In [50]:
phospho_pep[
  ,
  mod_pos_in_peptide := regexpr("[sty]", peptide_candidate)
]

phospho_pep[
  ,
  mod_aa := toupper(
    substr(
      peptide_candidate,
      mod_pos_in_peptide,
      mod_pos_in_peptide
    )
  )
]

phospho_pep[
  ,
  peptide_sequence := toupper(peptide_candidate)
]

phospho_pep[
  ,
  modified_peptide_sequence := peptide_candidate
]

head(phospho_pep)

row_id,gene_id,refseq_id,refseq_phosphosite,peptide_candidate,n_mods,mod_pos_in_peptide,mod_aa,peptide_sequence,modified_peptide_sequence
<int>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>
1,A2M,NP_000005.2,S928,ETTFNSLLCPSGGEVsEELSLK,1,16,S,ETTFNSLLCPSGGEVSEELSLK,ETTFNSLLCPSGGEVsEELSLK
2,ACADVL,NP_000009.1,S489,ELSGLGsALK,1,7,S,ELSGLGSALK,ELSGLGsALK
3,ACAT1,NP_000010.1,Y170,GSTPyGGVK,1,5,Y,GSTPYGGVK,GSTPyGGVK
4,ACVRL1,NP_000011.2,S160,GLHSELGEsSLILK,1,9,S,GLHSELGESSLILK,GLHSELGEsSLILK
5,ACVRL1,NP_000011.2,S161,GLHSELGESsLILK,1,10,S,GLHSELGESSLILK,GLHSELGESsLILK
6,ACVRL1,NP_000011.2,S495,ISNsPEKPK,1,4,S,ISNSPEKPK,ISNsPEKPK


In [51]:
fasta_lines <- readLines("tools/references/library.fasta")

header_idx <- grep("^>", fasta_lines)

fasta_dt <- data.table(
  header = fasta_lines[header_idx],
  start = header_idx + 1,
  end = c(header_idx[-1] - 1, length(fasta_lines))
)

fasta_dt[
  ,
  sequence := vapply(
    seq_len(.N),
    function(i) paste0(fasta_lines[start[i]:end[i]], collapse = ""),
    character(1)
  )
]

fasta_dt[
  ,
  protein_id_uniprot := sub("^>[^|]*\\|([^|]+)\\|.*", "\\1", header)
]

fasta_dt[
  ,
  gene_id := ifelse(
    grepl(" GN=", header),
    sub(".* GN=([^ ]+).*", "\\1", header),
    NA_character_
  )
]

fasta_dt <- fasta_dt[!is.na(gene_id)]
fasta_dt[, is_sp := grepl("^>sp\\|", header)]

head(fasta_dt)

header,start,end,sequence,protein_id_uniprot,gene_id,is_sp
<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<lgl>
>sp|Q6ZSK4|NTAS1_HUMAN Putative uncharacterized protein NTM-AS1 OS=Homo sapiens OX=9606 GN=NTM-AS1 PE=5 SV=1,2,4,MKGQEGIRGEGCTDPEIKASPQMWAARFRGMRSRFSPLFSQATEMGPRVSAGWCLSGGGRKVSSLQGDFPPGGFWALSNDSALSLPPLSLPHPHPLRPPGLGVNEFTQGLHPPLHPAASVFQTCFYRKPHYCSTLRPTTT,Q6ZSK4,NTM-AS1,TRUE
>sp|Q9Y263|PLAP_HUMAN Phospholipase A-2-activating protein OS=Homo sapiens OX=9606 GN=PLAA PE=1 SV=2,6,19,MTSGATRYRLSCSLRGHELDVRGLVCCAYPPGAFVSVSRDRTTRLWAPDSPNRSFTEMHCMSGHSNFVSCVCIIPSSDIYPHGLIATGGNDHNICIFSLDSPMPLYILKGHKNTVCSLSSGKFGTLLSGSWDTTAKVWLNDKCMMTLQGHTAAVWAVKILPEQGLMLTGSADKTVKLWKAGRCERTFSGHEDCVRGLAILSETEFLSCANDASIRRWQITGECLEVYYGHTNYIYSISVFPNCRDFVTTAEDRSLRIWKHGECAQTIRLPAQSIWCCCVLDNGDIVVGASDGIIRVFTESEDRTASAEEIKAFEKELSHATIDSKTGDLGDINAEQLPGREHLNEPGTREGQTRLIRDGEKVEAYQWSVSEGRWIKIGDVVGSSGANQQTSGKVLYEGKEFDYVFSIDVNEGGPSYKLPYNTSDDPWLTAYNFLQKNDLNPMFLDQVAKFIIDNTKGQMLGLGNPSFSDPFTGGGRYVPGSSGSSNTLPTADPFTGAGRYVPGSASMGTTMAGVDPFTGNSAYRSAASKTMNIYFPKKEAVTFDQANPTQILGKLKELNGTAPEEKKLTEDDLILLEKILSLICNSSSEKPTVQQLQILWKAINCPEDIVFPALDILRLSIKHPSVNENFCNEKEGAQFSSHLINLLNPKGKPANQLLALRTFCNCFVGQAGQKLMMSQRESLMSHAIELKSGSNKNIHIALATLALNYSVCFHKDHNIEGKAQCLSLISTILEVVQDLEATFRLLVALGTLISDDSNAVQLAKSLGVDSQIKKYSSVSEPAKVSECCRFILNLL,Q9Y263,PLAA,TRUE
>sp|Q96RE7|NACC1_HUMAN Nucleus accumbens-associated protein 1 OS=Homo sapiens OX=9606 GN=NACC1 PE=1 SV=1,21,29,MAQTLQMEIPNFGNSILECLNEQRLQGLYCDVSVVVKGHAFKAHRAVLAASSSYFRDLFNNSRSAVVELPAAVQPQSFQQILSFCYTGRLSMNVGDQFLLMYTAGFLQIQEIMEKGTEFFLKVSSPSCDSQGLHAEEAPSSEPQSPVAQTSGWPACSTPLPLVSRVKTEQQESDSVQCMPVAKRLWDSGQKEAGGGGNGSRKMAKFSTPDLAANRPHQPPPPQQAPVVAAAQPAVAAGAGQPAGGVAAAGGVVSGPSTSERTSPGTSSAYTSDSPGSYHNEEDEEEDGGEEGMDEQYRQICNMYTMYSMMNVGQTAEKVEALPEQVAPESRNRIRVRQDLASLPAELINQIGNRCHPKLYDEGDPSEKLELVTGTNVYITRAQLMNCHVSAGTRHKVLLRRLLASFFDRNTLANSCGTGIRSSTNDPRRKPLDSRVLHAVKYYCQNFAPNFKESEMNAIAADMCTNARRVVRKSWMPKVKVLKAEDDAYTTFISETGKIEPDMMGVEHGFETASHEGEAGPSAEALQ,Q96RE7,NACC1,TRUE
>sp|O43312|MTSS1_HUMAN Protein MTSS 1 OS=Homo sapiens OX=9606 GN=MTSS1 PE=1 SV=2,31,43,MEAVIEKECSALGGLFQTIISDMKGSYPVWEDFINKAGKLQSQLRTTVVAAAAFLDAFQKVADMATNTRGGTREIGSALTRMCMRHRSIEAKLRQFSSALIDCLINPLQEQMEEWKKVANQLDKDHAKEYKKARQEIKKKSSDTLKLQKKAKKGRGDIQPQLDSALQDVNDKYLLLEETEKQAVRKALIEERGRFCTFISMLRPVIEEEISMLGEITHLQTISEDLKSLTMDPHKLPSSSEQVILDLKGSDYSWSYQTPPSSPSTTMSRKSSVCSSLNSVNSSDSRSSGSHSHSPSSHYRYRSSNLAQQAPVRLSSVSSHDSGFISQDAFQSKSPSPMPPEAPNQLSNGFSHYSLSSESHVGPTGAGLFPHCLPASRLLPRVTSVHLPDYAHYYTIGPGMFPSSQIPSWKDWAKPGPYDQPLVNTLQRRKEKREPDPNGGGPTTASGPPAAAEEAQRPRSMTVSAATRPGEEMEACEELALALSRGLQLDTQRSSRDSLQCSSGYSTQTTTPCCSEDTIPSQVSDYDYFSVSGDQEADQQEFDKSSTIPRNSDISQSYRRMFQAKRPASTAGLPTTLGPAMVTPGVATIRRTPSTKPSVRRGTIGAGPIPIKTPVIPVKTPTVPDLPGVLPAPPDGPEERGEHSPESPSVGEGPQGVTSMPSSMWSGQASVNPPLPGPKPSIPEEHRQAIPESEAEDQEREPPSATVSPGQIPESDPADLSPRDTPQGEDMLNAIRRGVKLKKTTTNDRSAPRFS,O43312,MTSS1,TRUE
>sp|Q9NP80|PLPL8_HUMAN Calcium-independent phospholipase A2-gamma OS=Homo sapiens OX=9606 GN=PNPLA8 PE=1 SV=1,45,58,MSINLTVDIYIYLLSNARSVCGKQRSKQLYFLFSPKHYWRISHISLQRGFHTNIIRCKWTKSEAHSCSKHCYSPSNHGLHIGILKLSTSAPKGLTKVNICMSRIKSTLNSVSKAVFGNQNEMISRLAQFKPSSQILRKVSDSGWLKQKNIKQAIKSLKKYSDKSAEKSPFPEEKSHIIDKEEDIGKRSLFHYTSSITTKFGDSFYFLSNHINSYFKRKEKMSQQKENEHFRDKSELEDKKVEEGKLRSPDPGILAYKPGSESVHTVDKPTSPSAIPDVLQVSTKQSIANFLSRPTEGVQALVGGYIGGLVPKLKYDSKSQSEEQEEPAKTDQAVSKDRNAEEKKRLSLQREKIIARVSIDNRTRALVQALRRTTDPKLCITRVEELTFHLLEFPEGKGVAVKERIIPYLLRLRQIKDETLQAAVREILALIGYVDPVKGRGIRILSIDGGGTRGVVALQTLRKLVELTQKPVHQLFDYICGVSTGAILAFMLGLFHMPLDECEELYRKLGSDVFSQNVIVGTVKMSWSHAFYDSQTWENILKDRMGSALMIETARNPTCPKVAAVSTIVNRGITPKAFVFRNYGHFPGINSHYLGGCQYKMWQAIRASSAAPGYFAEYALGNDLHQDGGLLLNNPSALAMHECKCLWPDVPLECIVSLGTGRYESDVRNTVTYTSLKTKLSNVINSATDTEEVHIMLDGLLPPDTYFRFNPVMCENIPLDESRNEKLDQLQLEGLKYIERNEQKMKKVAKILSQEKTTLQKINDWIKLKTDMYEGLPFFSKL,Q9NP80,PNPLA8,TRUE
">sp|Q15319|PO4F3_HUMAN POU domain, class 4, transcription factor 3 OS=Homo sapiens OX=9606 GN=POU4F3 PE=1 SV=1",60,65,MMAMNSKQPFGMHPVLQEPKFSSLHSGSEAMRRVCLPAPQLQGNIFGSFDESLLARAEALAAVDIVSHGKNHPFKPDATYHTMSSVPCTSTSSTVPISHPAALTSHPHHAVHQGLEGDLLEHISPTLSVSGLGAPEHSVMPAQIHPHHLGA

In [52]:
cand <- merge(
  phospho_pep,
  fasta_dt[
    ,
    .(
      gene_id,
      protein_id_uniprot,
      fasta_sequence = sequence,
      is_sp
    )
  ],
  by = "gene_id",
  allow.cartesian = TRUE
)

cand[
  ,
  peptide_start := mapply(
    function(pat, seq) {
      regexpr(pat, seq, fixed = TRUE)[1]
    },
    peptide_sequence,
    fasta_sequence
  )
]

cand <- cand[peptide_start > 0]

cat("candidate matches:", nrow(cand), "\n")
cat("mapped original rows:", uniqueN(cand$row_id), "\n")

candidate matches: 48880 
mapped original rows: 43351 


In [53]:
cand[
  ,
  uniprot_site_pos := peptide_start + mod_pos_in_peptide - 1
]

cand[
  ,
  uniprot_site_aa := substr(
    fasta_sequence,
    uniprot_site_pos,
    uniprot_site_pos
  )
]

cand <- cand[mod_aa == uniprot_site_aa]

cat("AA-consistent matches:", nrow(cand), "\n")
cat("AA-consistent original rows:", uniqueN(cand$row_id), "\n")

cand[
  ,
  phosphosite := paste0(
    uniprot_site_aa,
    uniprot_site_pos
  )
]

AA-consistent matches: 48880 
AA-consistent original rows: 43351 


In [54]:
setorder(cand, row_id, -is_sp, protein_id_uniprot)

cand_best <- cand[
  ,
  .SD[1],
  by = row_id
]

cat("final mapped phospho rows:", nrow(cand_best), "\n")

head(
  cand_best[
    ,
    .(
      gene_id,
      refseq_id,
      refseq_phosphosite,
      protein_id_uniprot,
      phosphosite,
      modified_peptide_sequence,
      peptide_sequence
    )
  ],
  20
)

final mapped phospho rows: 43351 


gene_id,refseq_id,refseq_phosphosite,protein_id_uniprot,phosphosite,modified_peptide_sequence,peptide_sequence
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
A2M,NP_000005.2,S928,P01023,S928,ETTFNSLLCPSGGEVsEELSLK,ETTFNSLLCPSGGEVSEELSLK
ACADVL,NP_000009.1,S489,P49748,S489,ELSGLGsALK,ELSGLGSALK
ACAT1,NP_000010.1,Y170,P24752,Y170,GSTPyGGVK,GSTPYGGVK
ACVRL1,NP_000011.2,S160,P37023,S160,GLHSELGEsSLILK,GLHSELGESSLILK
ACVRL1,NP_000011.2,S161,P37023,S161,GLHSELGESsLILK,GLHSELGESSLILK
ACVRL1,NP_000011.2,S495,P37023,S495,ISNsPEKPK,ISNSPEKPK
PSEN1,NP_000012.1,S366,P49768,S366,AAVQELSsSILAGEDPEER,AAVQELSSSILAGEDPEER
PSEN1,NP_000012.1,S367,P49768,S367,AAVQELSSsILAGEDPEER,AAVQELSSSILAGEDPEER
PSEN1,NP_000012.1,S43,P49768,S43,sLGHPEPLSNGRPQGNSR,SLGHPEPLSNGRPQGNSR


In [57]:
map_phospho <- cand_best[
  ,
  .(
    row_id,
    protein_id = protein_id_uniprot,
    phosphosite,
    modified_peptide_sequence,
    peptide_sequence
  )
]

drop_cols <- intersect(
  c("protein_id", "phosphosite", "site_id", "peptide_id",
    "modified_peptide_sequence", "peptide_sequence"),
  names(phospho)
)

phospho_base <- copy(phospho)

if (length(drop_cols) > 0) {
  phospho_base[, (drop_cols) := NULL]
}

phospho_mapped <- merge(
  phospho_base,
  map_phospho,
  by = "row_id",
  all.x = FALSE
)

phospho_mapped[, site_id := paste(gene_id, protein_id, phosphosite, sep = ":")]

phospho_mapped[
  ,
  peptide_id := paste(
    protein_id,
    phosphosite,
    modified_peptide_sequence,
    sep = "__"
  )
]

dim(phospho)
dim(phospho_mapped)

head(
  phospho_mapped[
    ,
    .(
      gene_id,
      refseq_id,
      refseq_phosphosite,
      protein_id,
      phosphosite,
      site_id,
      modified_peptide_sequence
    )
  ],
  20
)

[1] 51469   149

[1] 43351   153

gene_id,refseq_id,refseq_phosphosite,protein_id,phosphosite,site_id,modified_peptide_sequence
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
A2M,NP_000005.2,S928,P01023,S928,A2M:P01023:S928,ETTFNSLLCPSGGEVsEELSLK
ACADVL,NP_000009.1,S489,P49748,S489,ACADVL:P49748:S489,ELSGLGsALK
ACAT1,NP_000010.1,Y170,P24752,Y170,ACAT1:P24752:Y170,GSTPyGGVK
ACVRL1,NP_000011.2,S160,P37023,S160,ACVRL1:P37023:S160,GLHSELGEsSLILK
ACVRL1,NP_000011.2,S161,P37023,S161,ACVRL1:P37023:S161,GLHSELGESsLILK
ACVRL1,NP_000011.2,S495,P37023,S495,ACVRL1:P37023:S495,ISNsPEKPK
PSEN1,NP_000012.1,S366,P49768,S366,PSEN1:P49768:S366,AAVQELSsSILAGEDPEER
PSEN1,NP_000012.1,S367,P49768,S367,PSEN1:P49768:S367,AAVQELSSsILAGEDPEER
PSEN1,NP_000012.1,S43,P49768,S43,PSEN1:P49768:S43,sLGHPEPLSNGRPQGNSR


In [58]:
phospho_long <- melt(
  phospho_mapped,
  id.vars = c(
    "gene_id",
    "protein_id",
    "peptide_id",
    "site_id",
    "modified_peptide_sequence",
    "peptide_sequence",
    "phosphosite"
  ),
  measure.vars = sample_cols,
  variable.name = "run_id",
  value.name = "peptide_intensity",
  variable.factor = FALSE,
  na.rm = TRUE
)

phospho_long[, run_id := as.character(run_id)]
phospho_long[, peptide_intensity := as.numeric(peptide_intensity)]

phospho_long <- phospho_long[
  !is.na(peptide_intensity) & is.finite(peptide_intensity)
]

phospho_long <- phospho_long[
  ,
  .(
    gene_id,
    protein_id,
    peptide_id,
    site_id,
    modified_peptide_sequence,
    peptide_sequence,
    phosphosite,
    run_id,
    peptide_intensity
  )
]

dim(phospho_long)
head(phospho_long)

[1] 2272580       9

gene_id,protein_id,peptide_id,site_id,modified_peptide_sequence,peptide_sequence,phosphosite,run_id,peptide_intensity
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
ACADVL,P49748,P49748__S489__ELSGLGsALK,ACADVL:P49748:S489,ELSGLGsALK,ELSGLGSALK,S489,C3N-03884,13.84222
PSEN1,P49768,P49768__S367__AAVQELSSsILAGEDPEER,PSEN1:P49768:S367,AAVQELSSsILAGEDPEER,AAVQELSSSILAGEDPEER,S367,C3N-03884,18.93493
ALDOA,P04075,P04075__S132__GVVPLAGTNGETTTQGLDGLsER,ALDOA:P04075:S132,GVVPLAGTNGETTTQGLDGLsER,GVVPLAGTNGETTTQGLDGLSER,S132,C3N-03884,17.19216
ALDOA,P04075,P04075__S276__TVPPAVTGITFLSGGQsEEEASINLNAINK,ALDOA:P04075:S276,TVPPAVTGITFLSGGQsEEEASINLNAINK,TVPPAVTGITFLSGGQSEEEASINLNAINK,S276,C3N-03884,16.82438
ALDOA,P04075,P04075__S36__GILAADEsTGSIAK,ALDOA:P04075:S36,GILAADEsTGSIAK,GILAADESTGSIAK,S36,C3N-03884,23.08933
ALDOA,P04075,P04075__S39__GILAADESTGsIAKR,ALDOA:P04075:S39,GILAADESTGsIAKR,GILAADESTGSIAKR,S39,C3N-03884,22.97128


In [62]:
phospho_long_site <- phospho_long[
  ,
  .(
    gene_id = gene_id[1],
    protein_id = protein_id[1],
    peptide_id = site_id[1],
    modified_peptide_sequence = site_id[1],
    peptide_sequence = site_id[1],
    phosphosite = phosphosite[1],
    peptide_intensity = median(peptide_intensity, na.rm = TRUE)
  ),
  by = .(site_id, run_id)
]

phospho_long_site <- phospho_long_site[
  ,
  .(
    gene_id,
    protein_id,
    peptide_id,
    site_id,
    modified_peptide_sequence,
    peptide_sequence,
    phosphosite,
    run_id,
    peptide_intensity
  )
]

In [64]:
proteo <- fread("./data/cptac-pdac/proteomics_gene_level_MD_abundance_tumor.cct")

In [65]:
setnames(proteo, names(proteo)[1], "gene_id")

proteo[, gene_id := trimws(gene_id)]

sample_cols <- setdiff(names(proteo), "gene_id")

for (j in sample_cols) {
  set(
    proteo,
    i = which(!is.na(proteo[[j]]) & !is.finite(proteo[[j]])),
    j = j,
    value = NA_real_
  )
}

obs_n <- rowSums(!is.na(proteo[, ..sample_cols]))

summary(obs_n)

proteo <- proteo[obs_n > 0]

dim(proteo)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0    70.0   135.0   105.5   140.0   140.0 

[1] 11631   141

In [66]:
fasta_headers <- readLines("tools/references/library.fasta")
fasta_headers <- fasta_headers[grepl("^>", fasta_headers)]

protein_id <- sub("^>[^|]*\\|([^|]+)\\|.*", "\\1", fasta_headers)

gene_symbol <- ifelse(
  grepl(" GN=", fasta_headers),
  sub(".* GN=([^ ]+).*", "\\1", fasta_headers),
  NA_character_
)

map_dt <- data.table(
  gene_id = gene_symbol,
  protein_id = protein_id,
  fasta_header = fasta_headers
)

map_dt <- map_dt[!is.na(gene_id)]

map_dt[, is_sp := grepl("^>sp\\|", fasta_header)]
setorder(map_dt, gene_id, -is_sp)

map_dt <- map_dt[, .SD[1], by = gene_id]
map_dt <- map_dt[, .(gene_id, protein_id)]

In [67]:
proteo_annot <- merge(
  proteo,
  map_dt,
  by = "gene_id",
  all.x = TRUE
)

cat("genes in proteo:", nrow(proteo_annot), "\n")
cat("mapped genes:", sum(!is.na(proteo_annot$protein_id)), "\n")
cat("unmapped genes:", sum(is.na(proteo_annot$protein_id)), "\n")

head(proteo_annot[is.na(protein_id), gene_id], 50)

genes in proteo: 11631 
mapped genes: 11480 
unmapped genes: 151 


[1] "AAED1"           "ADGRE5"          "ADSS"            "ADSSL1"         
 [5] "AES"             "ANKHD1-EIF4EBP3" "APOBEC3A_B"      "ATP5MF-PTCD1"   
 [9] "ATP5S"           "BCL2L2-PABPN1"   "BUB1B-PAK6"      "C10orf88"       
[13] "C11orf58"        "C12orf43"        "C12orf75"        "C15orf38-AP3S2" 
[17] "C15orf48"        "C16orf45"        "C17orf49"        "C17orf97"       
[21] "C19orf33"        "C19orf38"        "C19orf66"        "C19orf70"       
[25] "C1orf116"        "C1orf123"        "C1orf35"         "C21orf2"        
[29] "C2orf54"         "C3orf52"         "C3orf58"         "C4B_2"          
[33] "C5orf15"         "C6orf106"        "C6orf203"        "C6orf222"       
[37] "C6orf58"         "C7orf43"         "C7orf55-LUC7L2"  "C8orf59"        
[41] "CENPS-CORT"      "COL4A3BP"        "COMMD3-BMI1"     "CORO7-PAM16"    
[45] "CTGF"            "CXorf36"         "CYR61"           "DEFA1B"         
[49] "DIRC2"           "F8A2"

In [68]:
proteo_annot <- proteo_annot[!is.na(protein_id)]

sample_cols <- setdiff(names(proteo_annot), c("gene_id", "protein_id"))

proteo_long <- melt(
  proteo_annot,
  id.vars = c("gene_id", "protein_id"),
  measure.vars = sample_cols,
  variable.name = "run_id",
  value.name = "peptide_intensity",
  variable.factor = FALSE,
  na.rm = FALSE
)

proteo_long[, run_id := as.character(run_id)]

proteo_long[, peptide_id := protein_id]
proteo_long[, modified_peptide_sequence := protein_id]
proteo_long[, peptide_sequence := protein_id]
proteo_long[, phosphosite := "PA"]
proteo_long[, site_id := paste(gene_id, protein_id, phosphosite, sep = ":")]

proteo_long <- proteo_long[
  ,
  .(
    gene_id,
    protein_id,
    peptide_id,
    site_id,
    modified_peptide_sequence,
    peptide_sequence,
    phosphosite,
    run_id,
    peptide_intensity
  )
]

dim(proteo_long)
head(proteo_long)

[1] 1607200       9

gene_id,protein_id,peptide_id,site_id,modified_peptide_sequence,peptide_sequence,phosphosite,run_id,peptide_intensity
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
A1BG,P04217,P04217,A1BG:P04217:PA,P04217,P04217,PA,C3L-03394,28.67421
A1CF,Q9NQ94,Q9NQ94,A1CF:Q9NQ94:PA,Q9NQ94,Q9NQ94,PA,C3L-03394,24.02035
A2M,P01023,P01023,A2M:P01023:PA,P01023,P01023,PA,C3L-03394,29.77424
A2ML1,A8K2U0,A8K2U0,A2ML1:A8K2U0:PA,A8K2U0,A8K2U0,PA,C3L-03394,NA
A4GALT,Q9NPC4,Q9NPC4,A4GALT:Q9NPC4:PA,Q9NPC4,Q9NPC4,PA,C3L-03394,NA
A4GNT,Q9UNA3,Q9UNA3,A4GNT:Q9UNA3:PA,Q9UNA3,Q9UNA3,PA,C3L-03394,NA


In [69]:
dir.create("./data/cptac-pdac/processed")

Warning message in dir.create("./data/cptac-pdac/processed"):
“'./data/cptac-pdac/processed' already exists”


In [70]:
saveRDS(phospho_long_site, "./data/cptac-pdac/processed/CPTAC_PDAC_phospho.rds")
saveRDS(proteo_long, "./data/cptac-pdac/processed/CPTAC_PDAC_proteo.rds")